<a href="https://colab.research.google.com/github/aapaul29/Distance-Recognition-SML1-/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SML Project 1
This notebook works in both **VS Code (local)** and **Google Colab**.

## 1. Environment Detection & Setup

In [8]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules
print(f"Running in {'Google Colab' if IN_COLAB else 'VS Code / local'}")

Running in VS Code / local


## 2. Google Colab: Mount Drive & Navigate to Project
This cell only runs on Colab. Adjust `COLAB_PROJECT_PATH` to where you uploaded the project folder in your Google Drive.

In [9]:
# ---------------------------------------------------------------
# COLAB ONLY — skip if running locally
# Adjust this path to match your Google Drive folder structure.
COLAB_PROJECT_PATH = "/content/drive/MyDrive/Aaryan/Uni/BS Mech Eng ETH Zurich/2nd year/Semester 4/Stochastics & Machine Learning/Distance-Recognition-SML1-"
# ---------------------------------------------------------------

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd $COLAB_PROJECT_PATH
    print(f"Working directory: {os.getcwd()}")

    # Unzip data if not already extracted
    data_zip = os.path.join(COLAB_PROJECT_PATH, "data.zip")
    data_dir = os.path.join(COLAB_PROJECT_PATH, "data")
    if os.path.exists(data_zip) and not os.path.exists(data_dir):
        print("Unzipping data...")
        os.makedirs(data_dir, exist_ok=True)
        !unzip -q $data_zip -d $data_dir
        print("Done.")
    else:
        print("Data directory already exists or no zip found — skipping unzip.")
else:
    print("Local environment — no mounting needed.")

Local environment — no mounting needed.


## 3. Install / Verify Dependencies

In [10]:
# On Colab, install any missing packages from requirements.txt.
# Locally, this is a no-op if your conda env is already set up.
if IN_COLAB:
    !pip install -q -r requirements.txt

## 4. Auto-reload & Imports

In [11]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from utils import load_config, load_dataset, load_test_dataset, print_results, save_results

# Add further sklearn imports here as needed.
# Note: SVRs are NOT allowed in this project.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 5. Load Config & Data

In [12]:
config = load_config()

images, distances = load_dataset(config, split="train")
print(f"[INFO]: Dataset loaded with {len(images)} samples.")
print(f"[INFO]: Feature dimensionality: {images.shape[1]}")

[INFO]: Configs are loaded with: 
 {'data_dir': PosixPath('data'), 'load_rgb': False, 'downsample_factor': 2}
[INFO]: Dataset loaded with 3000 samples.
[INFO]: Feature dimensionality: 22500


## 6. Your Implementation
Implement preprocessing, model training, and evaluation below.

In [13]:
# Preprocessing & Feature Engineering
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.ndimage import sobel, uniform_filter, convolve
import numpy as np

ds = config["downsample_factor"]
H = W = 300 // ds
C = 3 if config["load_rgb"] else 1
print(f"Image size: {H}x{W}, channels: {C}")

# ─── Gabor kernel generation (manual, no skimage needed) ───

def make_gabor_kernel(frequency, theta, sigma_x=3.0, sigma_y=3.0, size=9):
    """Create a Gabor filter kernel using numpy only."""
    half = size // 2
    y, x = np.mgrid[-half:half+1, -half:half+1]
    x_theta = x * np.cos(theta) + y * np.sin(theta)
    y_theta = -x * np.sin(theta) + y * np.cos(theta)
    gb = np.exp(-0.5 * (x_theta**2 / sigma_x**2 + y_theta**2 / sigma_y**2))
    gb *= np.cos(2 * np.pi * frequency * x_theta)
    return gb.astype(np.float32)

# Pre-compute Gabor kernels
GABOR_KERNELS = []
for freq in [0.05, 0.1, 0.2, 0.3]:
    for theta in [0, np.pi/4, np.pi/2, 3*np.pi/4]:
        GABOR_KERNELS.append(make_gabor_kernel(freq, theta))

# ─── Feature extraction functions ───

def extract_lbp(img_2d):
    """Local Binary Patterns histogram."""
    H_, W_ = img_2d.shape
    lbp = np.zeros((H_, W_), dtype=np.uint8)
    center = img_2d[1:-1, 1:-1]
    neighbors = [
        img_2d[0:-2, 0:-2], img_2d[0:-2, 1:-1], img_2d[0:-2, 2:],
        img_2d[1:-1, 2:],   img_2d[2:, 2:],     img_2d[2:, 1:-1],
        img_2d[2:, 0:-2],   img_2d[1:-1, 0:-2]
    ]
    for i, n in enumerate(neighbors):
        lbp[1:-1, 1:-1] += (n >= center).astype(np.uint8) * (2**i)
    hist, _ = np.histogram(lbp, bins=32, range=(0, 256))
    return hist.astype(np.float32) / (hist.sum() + 1e-8)

def extract_handcrafted_features(X, H, W):
    """Extract comprehensive handcrafted features."""
    all_feat = []
    for img in X:
        img_2d = img.reshape(H, W).astype(np.float32)
        weights = np.linspace(0.3, 1.0, H).reshape(-1, 1)
        img_w = img_2d * weights
        feats = []

        # --- A. Row-wise statistics (50 strips) ---
        n_strips = 50
        strip_h = H // n_strips
        row_means = []
        for i in range(n_strips):
            strip = img_w[i*strip_h:(i+1)*strip_h, :]
            m = strip.mean()
            row_means.append(m)
            feats.extend([m, strip.std(), strip.min(), strip.max()])
        for i in range(1, n_strips):
            feats.append(row_means[i] - row_means[i-1])

        # --- B. Multi-scale spatial pyramid (2x2, 4x4, 8x8, 16x16) ---
        for grid_size in [2, 4, 8, 16]:
            gh, gw = H // grid_size, W // grid_size
            for i in range(grid_size):
                for j in range(grid_size):
                    cell = img_w[i*gh:(i+1)*gh, j*gw:(j+1)*gw]
                    feats.extend([cell.mean(), cell.std()])

        # --- C. Edge / floor features ---
        sx = sobel(img_w, axis=1)
        sy = sobel(img_w, axis=0)
        feats.extend([sx.mean(), sx.var(), sy.mean(), sy.var()])
        hist_sx, _ = np.histogram(sx, bins=16)
        hist_sy, _ = np.histogram(sy, bins=16)
        feats.extend((hist_sx / (hist_sx.sum() + 1e-8)).tolist())
        feats.extend((hist_sy / (hist_sy.sum() + 1e-8)).tolist())

        grad_map = np.abs(np.diff(img_2d, axis=0))
        for threshold in [5, 10, 15, 20, 25, 35, 50]:
            strong_edges = grad_map > threshold
            rows_idx = np.where(strong_edges, np.arange(H-1).reshape(-1, 1), -1)
            max_rows = rows_idx.max(axis=0)
            floor_positions = np.where(max_rows >= 0, max_rows / H, 0).astype(np.float32)
            valid = floor_positions[floor_positions > 0]
            if len(valid) > 0:
                feats.extend([np.median(valid), np.percentile(valid, 25),
                              np.percentile(valid, 75), np.mean(valid), np.std(valid)])
            else:
                feats.extend([0, 0, 0, 0, 0])

        # Strip gradient magnitudes (16 strips)
        strip_h16 = H // 16
        for i in range(16):
            strip = img_w[i*strip_h16:(i+1)*strip_h16, :]
            sx_s = sobel(strip, axis=1)
            sy_s = sobel(strip, axis=0)
            mag = np.sqrt(sx_s**2 + sy_s**2)
            feats.extend([mag.mean(), mag.var()])

        # --- D. Global statistics ---
        hist_global, _ = np.histogram(img_2d, bins=32, range=(0, 255))
        feats.extend((hist_global / (hist_global.sum() + 1e-8)).tolist())
        for p in [5, 10, 25, 50, 75, 90, 95]:
            feats.append(np.percentile(img_2d, p))
        top_mean = img_2d[:H//2, :].mean()
        bot_mean = img_2d[H//2:, :].mean()
        feats.append(bot_mean / (top_mean + 1e-8))
        feats.extend([img_2d.mean(), img_2d.std()])
        # Column-wise minimums (subsampled for ds=1)
        feats.extend(np.min(img_w, axis=0)[::3].tolist())

        # --- E. LBP ---
        feats.extend(extract_lbp(img_2d).tolist())

        # --- F. Gabor filter responses (NEW) ---
        for kernel in GABOR_KERNELS:
            response = convolve(img_2d, kernel)
            feats.extend([response.mean(), response.var(),
                          np.percentile(np.abs(response), 90)])

        # --- G. FFT features (NEW) ---
        fft = np.fft.fft2(img_2d)
        fft_mag = np.abs(np.fft.fftshift(fft))
        cy, cx = H // 2, W // 2
        Y, X_grid = np.ogrid[:H, :W]
        r = np.sqrt((Y - cy)**2 + (X_grid - cx)**2).astype(int)
        max_r = min(cy, cx)
        n_bands = 8
        band_width = max_r // n_bands
        for b in range(n_bands):
            mask = (r >= b * band_width) & (r < (b + 1) * band_width)
            if mask.any():
                feats.append(fft_mag[mask].mean())
            else:
                feats.append(0)

        # --- H. Multi-scale blur features (NEW) ---
        for scale in [3, 7, 15, 25]:
            blurred = uniform_filter(img_2d, size=scale)
            diff = img_2d - blurred
            feats.extend([diff.mean(), diff.var(), np.abs(diff).mean()])

        # --- I. Vertical intensity profile (NEW) ---
        col_means = img_2d.mean(axis=1)
        v_grad = np.diff(col_means)
        feats.extend([v_grad.mean(), v_grad.std(), v_grad.min(), v_grad.max()])
        feats.append(np.argmax(np.abs(v_grad)) / H)
        feats.append(np.argmin(v_grad) / H)
        for p in [10, 25, 50, 75, 90]:
            feats.append(np.percentile(col_means, p))

        all_feat.append(np.array(feats, dtype=np.float32))
    return np.array(all_feat)

def augment(X, y, H, W, C):
    """Data augmentation: flip, brightness, blur, noise, contrast."""
    X_aug, y_aug = [X], [y]
    X_flip = X.reshape(-1, H, W, C)[:, :, ::-1, :].reshape(len(X), -1)
    X_aug.append(X_flip); y_aug.append(y)
    X_dark = np.clip(X * 0.75, 0, 255)
    X_aug.append(X_dark); y_aug.append(y)
    X_bright = np.clip(X * 1.25, 0, 255)
    X_aug.append(X_bright); y_aug.append(y)
    X_blur = []
    for img in X:
        img_2d = img.reshape(H, W).astype(np.float32)
        blurred = uniform_filter(img_2d, size=3).reshape(-1)
        X_blur.append(blurred)
    X_aug.append(np.array(X_blur)); y_aug.append(y)
    rng = np.random.RandomState(42)
    noise = rng.normal(0, 5, X.shape)
    X_noisy = np.clip(X + noise, 0, 255)
    X_aug.append(X_noisy); y_aug.append(y)
    # Contrast augmentation
    means = X.mean(axis=1, keepdims=True)
    X_contrast = np.clip((X - means) * 1.3 + means, 0, 255)
    X_aug.append(X_contrast); y_aug.append(y)
    return np.vstack(X_aug), np.concatenate(y_aug)

# ─── Prepare data ───
X_train_raw, X_val_raw, y_train_raw, y_val = train_test_split(
    images, distances, test_size=0.15, random_state=42
)

print("Fitting PCA on training data...")
weights_pca = np.linspace(0.3, 1.0, H).reshape(1, -1)
weights_full = np.tile(weights_pca, (1, W))
X_train_weighted = X_train_raw * weights_full

pca_full = PCA(n_components=500, random_state=42)
pca_full.fit(X_train_weighted)
print(f"PCA full: {pca_full.explained_variance_ratio_.sum()*100:.1f}% variance in 500 components")

# Bottom-third PCA (floor area = strongest distance cue)
bottom_start_row = (2 * H // 3)
X_train_bottom = X_train_raw.reshape(-1, H, W)[:, bottom_start_row:, :].reshape(len(X_train_raw), -1)
pca_bottom = PCA(n_components=150, random_state=42)
pca_bottom.fit(X_train_bottom)
print(f"PCA bottom-third: {pca_bottom.explained_variance_ratio_.sum()*100:.1f}% variance in 150 components")

def build_features(X_raw, pca_full, pca_bottom, H, W):
    """Combine PCA features + handcrafted features."""
    weights_full = np.tile(np.linspace(0.3, 1.0, H).reshape(1, -1), (1, W))
    X_pca_full = pca_full.transform(X_raw * weights_full)
    bottom_start = (2 * H // 3)
    X_bottom = X_raw.reshape(-1, H, W)[:, bottom_start:, :].reshape(len(X_raw), -1)
    X_pca_bottom = pca_bottom.transform(X_bottom)
    X_hand = extract_handcrafted_features(X_raw, H, W)
    return np.hstack([X_pca_full, X_pca_bottom, X_hand])

# Augment
X_aug, y_train = augment(X_train_raw, y_train_raw, H, W, C)
print(f"Augmented training samples: {len(X_aug)}")

print("Extracting features for training set...")
X_train_feat = build_features(X_aug, pca_full, pca_bottom, H, W)
print("Extracting features for validation set...")
X_val_feat = build_features(X_val_raw, pca_full, pca_bottom, H, W)

scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_val_feat = scaler.transform(X_val_feat)

print(f"Train features: {X_train_feat.shape}, Val features: {X_val_feat.shape}")

Image size: 150x150, channels: 1
Fitting PCA on training data...
PCA full: 97.7% variance in 500 components
PCA bottom-third: 97.0% variance in 150 components
Augmented training samples: 17850
Extracting features for training set...
Extracting features for validation set...
Train features: (17850, 1885), Val features: (450, 1885)


In [14]:
# Model definition & training
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, StackingRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

def build_stacking_model(seed=0):
    """Build a diverse stacking ensemble with tuned hyperparameters."""
    estimators = [
        ("hgb_mae", HistGradientBoostingRegressor(
            max_iter=1500, max_depth=10, learning_rate=0.02,
            min_samples_leaf=15, loss='absolute_error',
            l2_regularization=0.1, random_state=seed
        )),
        ("hgb_mse", HistGradientBoostingRegressor(
            max_iter=1000, max_depth=8, learning_rate=0.03,
            min_samples_leaf=20, loss='squared_error',
            l2_regularization=0.1, random_state=seed
        )),
        ("hgb_huber", HistGradientBoostingRegressor(
            max_iter=800, max_depth=7, learning_rate=0.04,
            min_samples_leaf=25, loss='absolute_error',
            l2_regularization=0.5, random_state=seed + 100
        )),
        ("et", ExtraTreesRegressor(
            n_estimators=800, max_depth=30, max_features=0.5,
            n_jobs=-1, random_state=seed
        )),
        ("ridge", Ridge(alpha=1.0)),
        ("knn", KNeighborsRegressor(n_neighbors=12, weights='distance')),
    ]
    return StackingRegressor(
        estimators=estimators,
        final_estimator=Ridge(alpha=10.0),
        cv=5, n_jobs=-1
    )

# Train multi-seed ensemble (3 seeds for speed)
print("Training multi-seed stacking ensemble...")
models = []
val_preds_list = []

for seed in range(3):
    print(f"  Seed {seed}...")
    model = build_stacking_model(seed)
    model.fit(X_train_feat, y_train)
    val_pred = model.predict(X_val_feat)
    val_preds_list.append(val_pred)
    models.append(model)
    print(f"    MAE: {mean_absolute_error(y_val, val_pred)*100:.2f} cm")

val_pred_ensemble = np.mean(val_preds_list, axis=0)
print(f"\nEnsemble MAE ({len(models)} seeds): {mean_absolute_error(y_val, val_pred_ensemble)*100:.2f} cm")

Training multi-seed stacking ensemble...
  Seed 0...
    MAE: 12.08 cm
  Seed 1...
    MAE: 12.11 cm
  Seed 2...
    MAE: 11.91 cm

Ensemble MAE (3 seeds): 11.92 cm


In [15]:
# Evaluation
print("=== Validation Set ===")
print_results(y_val, val_pred_ensemble)

# Training set evaluation (on augmented data)
train_pred = np.mean([m.predict(X_train_feat) for m in models], axis=0)
print("\n=== Training Set ===")
print_results(y_train, train_pred)

# Check overfitting gap
train_mae = mean_absolute_error(y_train, train_pred) * 100
val_mae = mean_absolute_error(y_val, val_pred_ensemble) * 100
print(f"\nOverfitting gap: {val_mae - train_mae:.2f} cm")

=== Validation Set ===
MAE: 11.916
R2: 83.487

=== Training Set ===
MAE: 3.641
R2: 98.624

Overfitting gap: 8.27 cm


## 7. Predict on Test Set & Save

In [17]:
# Quick prediction using existing val-trained models (no retrain)
print("Generating test predictions using existing models...")

test_images_raw = load_test_dataset(config)
X_test_raw = np.array(test_images_raw)

print("Extracting test features...")
X_test_feat = build_features(X_test_raw, pca_full, pca_bottom, H, W)
X_test_feat = scaler.transform(X_test_feat)

test_pred = np.mean([m.predict(X_test_feat) for m in models], axis=0)
test_pred = np.clip(test_pred, 0.1, 4.0)

save_results(test_pred)
print(f"Saved! Predictions range: {test_pred.min():.3f} - {test_pred.max():.3f} m")

Generating test predictions using existing models...
Extracting test features...
Saved! Predictions range: 0.297 - 2.787 m
